# chess-vi — phục vụ model từ Colab

Model 14B cần ~28GB VRAM, máy local 4GB không chạy được. Notebook này bày model
ra một endpoint kiểu OpenAI rồi mở tunnel, để **web demo chạy ở máy bạn** gọi vào.

```
[máy bạn]  trình duyệt → chessvi.serve.web → HTTP → [Colab]  chessvi.serve.colab_api → model
```

Trình duyệt không gọi thẳng sang Colab: `serve/web.py` gọi từ phía Python. Nhờ
vậy không vướng CORS, và URL Colab không lộ ra trang web.

**Runtime phải có GPU.** Runtime → Change runtime type → A100 cho bản 14B.

Chạy lần lượt từ trên xuống. Cell số 6 in ra đúng lệnh cần chạy ở máy bạn.

## 1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/chessvi"

## 2. Lấy code

Cùng repo với notebook train. Chạy lại cell này mỗi lần mở runtime mới.

In [ ]:
import os

REPO = "/content/ChessVi"

if not os.path.isdir(REPO):
    !git clone https://github.com/trantrien1/ChessVi.git {REPO}
!git -C {REPO} pull

%cd {REPO}
!pip install -q -e "."

# peft <-> torchao: is_torchao_available() NÉM ImportError khi torchao cũ hơn
# 0.16, thay vì trả False. Chỉ chạm tới khi model KHÔNG lượng tử hoá, tức đúng
# đường phục vụ bf16. Gỡ chứ không nâng: bản mới có thể kéo theo torch khác.
!pip uninstall -y -q torchao

## 3. Adapter

Trỏ tới thư mục LoRA sau T8. Mặc định là bản 14B trên Drive.

In [ ]:
BASE_MODEL = "Qwen/Qwen3-14B"
ADAPTER = f"{DRIVE_ROOT}/outputs/sft-14b"

assert os.path.isdir(ADAPTER), f"Không thấy adapter ở {ADAPTER}"
print(sorted(os.listdir(ADAPTER))[:10])

## 4. Tải cloudflared

Tunnel loại "quick" — không cần đăng ký, không cần token, cho một URL
`https://....trycloudflare.com` sống cùng phiên.

In [ ]:
!wget -q -O /content/cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
!/content/cloudflared --version

## 5. Chạy model + tunnel

Nạp 14B mất vài phút lần đầu (tải ~28GB), các lần sau nhanh vì đã cache.

Cell này chạy xong là **giữ nguyên hai tiến trình nền** — đừng đóng tab, đừng
chạy lại. Muốn dừng thì dùng cell 7.

In [ ]:
import re
import subprocess
import time
import urllib.request

PORT = 8001
LOG = "/content/api.log"

# Dọn tiến trình của lần chạy trước, nếu có.
!pkill -f chessvi.serve.colab_api 2>/dev/null; pkill -f cloudflared 2>/dev/null; true
time.sleep(1)

api = subprocess.Popen(
    ["python", "-m", "chessvi.serve.colab_api",
     "--model-path", BASE_MODEL, "--adapter", ADAPTER, "--port", str(PORT)],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT, cwd=REPO,
)

print("Đang nạp model…")
health = f"http://127.0.0.1:{PORT}/health"
for _ in range(180):  # tối đa 15 phút, đủ cho lần tải đầu
    if api.poll() is not None:
        print(open(LOG).read()[-3000:])
        raise SystemExit("Server chết khi khởi động — xem log ở trên")
    try:
        with urllib.request.urlopen(health, timeout=2) as r:
            print("Model sẵn sàng:", r.read().decode())
            break
    except Exception:
        time.sleep(5)
else:
    raise SystemExit("Quá 15 phút chưa sẵn sàng, xem " + LOG)

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

URL = None
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        URL = found.group(0)
        break
assert URL, "Không lấy được URL tunnel"

print("\n" + "=" * 66)
print("Chạy lệnh này ở MÁY BẠN (trong thư mục repo):\n")
print(f"  python -m chessvi.serve.web --llm remote --api-base {URL}/v1\n")
print("Rồi mở http://127.0.0.1:8000")
print("=" * 66)

## 6. Thử nhanh từ Colab

Không bắt buộc. Gọi thẳng endpoint để chắc model trả lời được trước khi sang máy mình.

In [ ]:
import json
import urllib.request

payload = {
    "model": "chessvi",
    "messages": [{"role": "user", "content": "Chào, bạn là ai?"}],
    "temperature": 0.3,
}
req = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req, timeout=180) as r:
    print(json.load(r)["choices"][0]["message"]["content"])

## 7. Dừng

Chạy khi xong việc, hoặc trước khi chạy lại cell 5.

In [ ]:
!pkill -f chessvi.serve.colab_api 2>/dev/null; pkill -f cloudflared 2>/dev/null; true
print("Đã dừng")

## Gặp lỗi

**`Không gọi được ...` ở máy bạn** — tunnel đã chết. Colab ngắt phiên là URL mất;
chạy lại cell 5 sẽ ra URL **mới**, phải khởi động lại `serve.web` với URL đó.

**Trả lời rất chậm** — mỗi câu hỏi là một lượt sinh trên 14B, cỡ 10–30 giây tuỳ
GPU. Không có batching vì chỉ một người dùng.

**`CUDA out of memory`** — runtime không phải A100. 14B bf16 cần ~28GB. Dùng
T4/L4 thì phải đổi sang adapter 4B (`outputs/sft`) và `BASE_MODEL =
"Qwen/Qwen3-4B"`, nhưng chất lượng thấp hơn nhiều: 4B chỉ đạt 12% accuracy và
51% nước hợp lệ, so với 35% và 87% của 14B.